# ConformEval tutorial

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/wyqmath/ConformEval/blob/main/examples/notebooks/ConformEval_tutorial.ipynb)

This tutorial demonstrates the ConformEval workflow for protein knot detection using ESM-2 8M as an example model.


## 1. Select the dataset

Set `FULL_BENCHMARK=True` for the knot dataset or `FULL_BENCHMARK=False` for the 20-sequence synthetic example.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor
import csv, gzip, json, math, os, platform, subprocess, sys, tempfile, time
import urllib.request

assert sys.version_info >= (3, 9), "Python >=3.9 is required"
FULL_BENCHMARK = True
SOURCE_REF = "main"
REPOSITORY = "wyqmath/ConformEval"
WORK_ROOT = Path(tempfile.mkdtemp(prefix="conformeval-notebook-"))
SOURCE_ROOT = WORK_ROOT / "repository"
SOURCE_ROOT.mkdir()
RUN_ROOT = WORK_ROOT / "runs"
RUN_ROOT.mkdir()
MODEL_CACHE = Path(os.environ.get("CONFORMEVAL_MODEL_CACHE", str(Path.home() / ".cache/huggingface/hub")))
STARTED_AT = datetime.now(timezone.utc).isoformat()
print("Mode:", "complete knot-presence benchmark" if FULL_BENCHMARK else "synthetic interface example")
print("Source:", SOURCE_REF)

## 2. Download code and data


In [ ]:
def fetch(url):
    for attempt in range(3):
        try:
            request = urllib.request.Request(url, headers={"User-Agent": "ConformEval-notebook"})
            with urllib.request.urlopen(request, timeout=60) as response:
                return response.read()
        except Exception:
            if attempt == 2:
                raise
            time.sleep(attempt + 1)

tree_url = f"https://api.github.com/repos/{REPOSITORY}/git/trees/{SOURCE_REF}?recursive=1"
tree = json.loads(fetch(tree_url))
assert not tree.get("truncated"), "GitHub returned a partial file tree"
data_prefix = "data/base/" if FULL_BENCHMARK else "examples/"
root_files = {"pyproject.toml", "README.md", "LICENSE"}
selected = [entry for entry in tree["tree"] if entry["type"] == "blob" and
            (entry["path"] in root_files or entry["path"].startswith(("src/", "configs/", data_prefix)))]
assert selected

def download_entry(entry):
    relative = Path(entry["path"])
    assert not relative.is_absolute() and ".." not in relative.parts
    content = fetch(f"https://raw.githubusercontent.com/{REPOSITORY}/{SOURCE_REF}/{relative.as_posix()}")
    destination = SOURCE_ROOT / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_bytes(content)
    return {"path": relative.as_posix(), "bytes": len(content)}

with ThreadPoolExecutor(max_workers=4) as executor:
    downloaded = list(executor.map(download_entry, selected))
(RUN_ROOT / "download_manifest.json").write_text(
    json.dumps({"source_ref": SOURCE_REF, "files": downloaded}, indent=2), encoding="utf-8")
print(f"Downloaded {len(downloaded)} files ({sum(x['bytes'] for x in downloaded):,} bytes)")

## 3. Install dependencies


In [ ]:
def run_step(name, arguments, cwd=SOURCE_ROOT):
    started = time.monotonic()
    completed = subprocess.run([str(arg) for arg in arguments], cwd=cwd,
                               stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    (RUN_ROOT / f"{name}.log").write_text(completed.stdout, encoding="utf-8")
    if completed.returncode:
        print(completed.stdout[-12000:])
        raise RuntimeError(f"{name} failed with exit code {completed.returncode}")
    print(f"{name}: PASS ({time.monotonic() - started:.2f} s)")

run_step("install", [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", ".[esm2]"])

## 4. Prepare sequences and download ESM-2

The model configuration selects layers 3 and 6, each with 320 features per residue.


In [ ]:
TASK = "KNOT-PRESENCE"
DATA_DIR = SOURCE_ROOT / ("data/base" if FULL_BENCHMARK else "examples/base")
MODEL_CONFIG = SOURCE_ROOT / "configs/models/esm2_8m.json"
PROTOCOL = SOURCE_ROOT / "configs/scoring.json"
example_flags = [] if FULL_BENCHMARK else ["--example"]

def cli(module, *arguments):
    return [sys.executable, "-m", "conformeval." + module, *map(str, arguments)]

run_step("prepare", cli("tasks", "--task", TASK, "--data-dir", DATA_DIR,
                        "--output-dir", RUN_ROOT / "inputs", *example_flags))
# The adapters module exposes download_main as a console entry point.
run_step("checkpoint", [sys.executable, "-c", "from conformeval.adapters import download_main; download_main()",
                         "--model-config", MODEL_CONFIG, "--cache-dir", MODEL_CACHE])
expected_sequences = 569 if FULL_BENCHMARK else 20
prepared = json.loads((RUN_ROOT / "inputs/run.json").read_text())
# Knot preparation selects 20 of the 30 example sequences.
assert prepared["unique_sequences"] == expected_sequences
print("Unique encoder inputs:", prepared["unique_sequences"])

## 5. Extract representations

Mean pooling produces one representation per sequence for each selected layer. Sequences longer than 1,022 residues are processed in windows with a 128-residue overlap.


In [ ]:
run_step("inference", cli("extract", "--model-config", MODEL_CONFIG,
                          "--input-fasta", RUN_ROOT / "inputs/sequences.fasta",
                          "--output-dir", RUN_ROOT / "features", "--cache-dir", MODEL_CACHE,
                          "--local-files-only", "--sequence-only", "--device", "cpu", "--threads", "4"))
manifest_path = RUN_ROOT / "features/representation_manifest.json"
features = json.loads(manifest_path.read_text())
assert features["output_dimension"] == 320
assert features["candidate_layers"] == [3, 6]
assert len(features["archives"]) == expected_sequences
print("Representations:", len(features["archives"]), "sequences; dimension", features["output_dimension"])

## 6. Train predictors

Training compares layers 3 and 6 and values of 0.1 and 1.0 for the regularization parameter `C`. The output shows AUROC for the model and baselines based on sequence length, amino-acid composition, nearest neighbors, and the training-label prior.


In [ ]:
run_step("training", cli("train", "--task", TASK, "--data-dir", DATA_DIR,
                         "--feature-manifest", manifest_path,
                         "--config", SOURCE_ROOT / "configs/readers/linear.json",
                         "--protocol", PROTOCOL, "--output-dir", RUN_ROOT / "readers", *example_flags))
training_metrics = json.loads((RUN_ROOT / "readers/metrics.json").read_text())
with gzip.open(RUN_ROOT / "readers/predictions.tsv.gz", "rt", encoding="utf-8", newline="") as handle:
    predictions = list(csv.DictReader(handle, delimiter="\t"))
nested = [row for row in predictions if row["arm"] == "PLM-nested"]
expected_samples = 734 if FULL_BENCHMARK else 20
assert len(nested) == expected_samples
assert len({row["sample_id"] for row in nested}) == expected_samples
assert len({row["outer_fold"] for row in nested}) == 5
for arm, metrics in sorted(training_metrics.items()):
    print(f"{arm:22s} AUROC {metrics['auroc']:.6f}")

## 7. Score predictions

Export predictions to `nested_predictions.tsv` and calculate AUROC and its 95% bootstrap confidence interval.


In [ ]:
submission = RUN_ROOT / "nested_predictions.tsv"
fields = ["task_id", "sample_id", "outer_fold", "score"]
with submission.open("w", encoding="utf-8", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fields, delimiter="\t", lineterminator="\n")
    writer.writeheader()
    writer.writerows({field: row[field] for field in fields} for row in nested)
run_step("rescoring", cli("evaluate_predictions", "--task", TASK, "--data-dir", DATA_DIR,
                          "--predictions", submission,
                          "--model-metadata", RUN_ROOT / "readers/model_metadata.json",
                          "--protocol", PROTOCOL, "--output-dir", RUN_ROOT / "rescored"))
rescored = json.loads((RUN_ROOT / "rescored/metrics.json").read_text())["metrics"][TASK]
original = training_metrics["PLM-nested"]
assert rescored["n_samples"] == expected_samples
assert math.isclose(rescored["auroc"], original["auroc"], rel_tol=0, abs_tol=1e-12)
assert rescored["group_bootstrap"]["ci95"] == original["group_bootstrap"]["ci95"]
print("Independent rescoring: PASS")
print("Nested-reader AUROC:", round(rescored["auroc"], 6))
print("95% binding-group bootstrap interval:", rescored["group_bootstrap"]["ci95"])

## 8. Save outputs

Outputs are stored in `RUN_ROOT`. The `summary.json` file summarizes the score, model configuration, and software versions.


In [ ]:
from importlib.metadata import version

record = {
    "status": "PASS", "project": "ConformEval", "source_ref": SOURCE_REF,
    "mode": "complete_knot_benchmark" if FULL_BENCHMARK else "synthetic_interface_example",
    "distribution": "conformeval", "package_version": version("conformeval"),
    "task": TASK, "model_id": features["model_id"], "model_revision": features["model_revision"],
    "started_at": STARTED_AT, "ended_at": datetime.now(timezone.utc).isoformat(),
    "platform": platform.platform(), "python": platform.python_version(), "device": "cpu",
    "dependencies": {name: version(name) for name in
                     ["numpy", "scipy", "scikit-learn", "torch", "transformers", "huggingface-hub"]},
    "unique_sequences": expected_sequences, "scored_samples": expected_samples,
    "groups": rescored["n_binding_groups"], "auroc": rescored["auroc"],
    "ci95": rescored["group_bootstrap"]["ci95"], "independent_rescoring_matches": True,
    "outputs": [path.relative_to(RUN_ROOT).as_posix() for path in
        [RUN_ROOT / "download_manifest.json", manifest_path, RUN_ROOT / "readers/run.json",
         RUN_ROOT / "readers/predictions.tsv.gz", submission, RUN_ROOT / "rescored/metrics.json"]],
}
(RUN_ROOT / "summary.json").write_text(json.dumps(record, indent=2), encoding="utf-8")
print(json.dumps(record, indent=2))
print("Outputs:", RUN_ROOT)